# Canonical FI names — `pkg_fi_canonical.ipynb`

**Why.** The FI name in the staging data (`cpty_fin_entity_name`, carried into `pkg_cpty_month` / `pkg_cpty_dim`) is not
one name per bank. 232 routing numbers carry 2–3 names, mostly acquisitions (COMERICA → FIFTH THIRD) and spelling
variants (GECU / GECU FEDERAL CREDIT UNION), plus typos (ALLLY BANK). Our bank metrics count distinct FI *names*, so:
- a customer paying two accounts under one routing number can count two banks;
- a routing number whose name changes shows up as "lost a bank, gained a bank" — false bank churn, in the
  `fin_out_n` / bank-churn family that drives the attrition model.

**What this builds** (small reference tables in cdi_tm; set `WRITE_TABLES = False` to only look):
1. `pkg_fi_routing` — one row per 9-digit routing number: canonical name = the name in the **latest** month the routing
   appears (acquired banks map to their current owner, consistently across history), spelling-normalised; plus every
   name seen, change type and switch month.
2. `pkg_fi_name_map` — raw FI name → canonical name, for ids without a routing prefix (name-hash ids on 3B/3C/7B/9B).
   Only names with one clear canonical name across the routing numbers they appear under are changed.

Spellings are unified **only within a routing number** (GECU and GECU FEDERAL CREDIT UNION on the same routing). Two
institutions whose names differ only by a legal word (SAFE CREDIT UNION in California, SAFE FEDERAL CREDIT UNION in
South Carolina) are never merged.

**How to use downstream:** canonical FI of a counterparty = `pkg_fi_routing.fi_canonical` via its routing prefix, else
`pkg_fi_name_map.fi_canonical` via its raw FI name, else the raw name. §5 has the join.

**Impact (§4):** for one month, the number of distinct banks per customer and direction (all rails, and ACH-only out =
the attrition `fin_out_n` definition) and month-over-month lost banks, with raw vs canonical names.

In [ ]:
# §0  Setup
import re, time, warnings, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("pkg_fi_canonical")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", "2000")

DB           = "bdahd01p_dlcdi1_cdi_tm"
T_CPTY_MONTH = f"{DB}.pkg_cpty_month"
T_CPTY_DIM   = f"{DB}.pkg_cpty_dim"
T_EDGE_M     = f"{DB}.pkg_edge_monthly"
T_CUST_DIM   = f"{DB}.pkg_cust_dim"
T_FI_ROUTING = f"{DB}.pkg_fi_routing"
T_FI_NAMEMAP = f"{DB}.pkg_fi_name_map"
WRITE_TABLES = True

IMPACT_MONTH = "2026-08"          # §4 compares this month with the previous one
# Raw FI name → corrected name: typos, and renames §4b finds that no routing number explains (raw → new name)
FI_FIXES = {"ALLLY BANK": "ALLY BANK"}
# §4b history check over all months (customer × direction × bank, cached per month in scratch, resumable)
HIST_TM_ONLY     = True     # TM customers only (main population; keeps the cache small)
HIST_REBUILD     = False    # True: rebuild the cache (after the edge tables or pkg_cpty_month were rebuilt)
SWAP_MIN_CUST    = 50       # list bank swaps X → Y made by at least this many customer links in one month
SWAP_MIN_SHARE   = 0.2      # … that are at least this share of X's customer links the month before
SWAP_MAX_CHANGES = 5        # customer links that lose or gain more banks than this in a month are left out of swaps
# Fintech partner names: "<brand> WITH / PARTNERING WITH / POWERED BY <bank>" → the bank holding the account
FI_PARTNER_RE = (r"^.+?\s+(?:WITH|PARTNERING WITH|POWERED BY|IN PARTNERSHIP WITH|THROUGH)\s+"
                 r"(.*\b(?:BANK|CREDIT UNION|TRUST|N A|NA|FSB|SSB|SAVINGS|BANCORP|FCU)\b.*)$")
# Division names: "<brand> ADO / A DIVISION OF <parent>" → the parent bank (ACCORDIA BANK ADO QUILL BANK → QUILL BANK).
# A parent cut short by the feed (…A DIVISION OF TOWNEBAN) is completed when exactly one known name starts with it.
FI_DIVISION_RE = r"^.+?\s+(?:ADO|A DIVISION OF|DIVISION OF|A DIV OF|DIV OF)\s+(.{4,})$"
# Routing numbers whose names should NOT be unified (pairs you checked and found unrelated, see §2b)
FI_KEEP_RAW_ROUTINGS = []
# A raw FI name without a routing number (name-hash ids) takes the canonical name of the routing numbers it appears
# under only when one canonical name holds at least this share of its dollars there; otherwise it stays as is
# (FIRSTBANK is FirstBank Colorado → PNC, but also FirstBank Tennessee)
NAME_MIN_SHARE = 0.9
# Words ignored when testing whether two bank names are related (§2b)
FI_GENERIC = {"BANK", "BANKS", "NATIONAL", "TRUST", "COMPANY", "CREDIT", "UNION", "FEDERAL", "SAVINGS", "FINANCIAL",
              "COMMUNITY", "FIRST", "STATE", "CITIZENS", "PEOPLES", "AMERICAN", "UNITED", "SECURITY", "THE", "OF", "AND",
              "NA", "FSB", "SSB", "INC", "CORP", "CORPORATION", "EMPLOYEES", "BANCORP", "BANKING", "ASSOCIATION"}
# Words dropped to group spellings of one bank (GECU FEDERAL CREDIT UNION = GECU; WEBSTER BANK N A = WEBSTER BANK)
FI_STRIP = ["THE", "NA", "NATIONAL ASSOCIATION", "INC", "CORP", "CORPORATION", "CO", "COMPANY", "FSB", "SSB",
            "FEDERAL CREDIT UNION", "CREDIT UNION", "FCU", "LLC"]

def fi_fix(s):
    # typo fixes, then fintech partner names → the partner bank
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return s
    s = FI_FIXES.get(s, s)
    clean = re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", str(s).upper())).strip()
    m = re.match(FI_PARTNER_RE, clean) or re.match(FI_DIVISION_RE, clean)
    return m.group(1).strip() if m else s

def complete_truncated(fixed, known):
    # a fixed name that is no raw name anywhere but the start of exactly one raw name → that name
    known = sorted(set(known))
    out = {}
    for f in set(fixed) - set(known):
        if f is None or (isinstance(f, float) and np.isnan(f)) or len(str(f)) < 6:
            continue
        hits = [k for k in known if k.startswith(f + "") and k != f]
        if len(hits) == 1:
            out[f] = hits[0]
    return out

def fi_tokens(s):
    return {t for t in re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).split() if len(t) >= 3 and t not in FI_GENERIC}

def fi_key(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    k = re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()
    k = re.sub(r"\b([A-Z]) (?=[A-Z]\b)", r"\1", k)                    # N A → NA
    for w in sorted(FI_STRIP, key=len, reverse=True):
        k = re.sub(rf"\b{w}\b", " ", k)
    k = re.sub(r"\s+", " ", k).strip()
    return k or None

def prev_month(m):
    y, mm = map(int, m.split("-"))
    return f"{y - (mm == 1):04d}-{(mm - 2) % 12 + 1:02d}"

# ---- display ----
def _fmt(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if kind == "pct":
        return "0" if abs(v) < 5e-7 else (f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}")
    return f"{v:,.0f}" if kind == "int" else f"{v:,.1f}" if kind == "f1" else f"{v:,.2f}"

def fmt_df(df):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if not pd.api.types.is_numeric_dtype(s):
            out[c] = s.where(s.notna(), "")
            continue
        kind = ("pct" if "%" in str(c) else "f1" if "$M" in str(c) or "$B" in str(c)
                else "int" if (s.dropna() % 1 == 0).all() else "f2")
        out[c] = [_fmt(v, kind) for v in s]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#8a8984;margin:2px 0}</style>")

def show(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        else:
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1]).to_html(index=False, escape=True, border=0, classes="pkgt"))
    display(HTML(_CSS + "<div style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>" + "".join(parts) + "</div>"))

def note(t):
    return ("__note__", t)

ddt = dict(spark.table(T_CPTY_DIM).dtypes)
DIM_FI = next(c for c in ["fi", "cpty_fin_entity_name", "dominant_fi"] if c in ddt)
print(f"Spark {spark.version} | dim FI column: {DIM_FI} | impact month {IMPACT_MONTH} vs {prev_month(IMPACT_MONTH)}")

## §1 Routing × month × FI name (small: collected to pandas)

In [ ]:
# §1  Every FI name seen under each routing number, by month
cm = spark.table(T_CPTY_MONTH)
rt_raw = (cm.where(F.col("fi").isNotNull())
            .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
            .where(F.col("routing") != "")
            .groupBy("routing", "month", "fi")
            .agg(F.sum(F.col("amount").cast("double")).alias("amt"), F.countDistinct("cpty_id").alias("ids"))
            .toPandas())
rt_raw["fi_fixed"] = rt_raw.fi.map(fi_fix)
rt_raw["key"] = rt_raw.fi_fixed.map(fi_key)

# all raw FI names: dominant FI in the dim (with id counts) + every FI name in pkg_cpty_month (the monthly FI used
# by the counterparty metrics), including names only seen on ids without a routing prefix
dim_names = (spark.table(T_CPTY_DIM).where(F.col(DIM_FI).isNotNull())
                  .groupBy(F.col(DIM_FI).alias("fi")).agg(F.count("*").alias("ids")).toPandas())
cm_names = cm.where(F.col("fi").isNotNull()).select("fi").distinct().toPandas()
dim_names = dim_names.merge(cm_names, on="fi", how="outer").fillna({"ids": 0})
dim_names["ids"] = dim_names.ids.astype("int64")
dim_names["fi_fixed"] = dim_names.fi.map(fi_fix)
# complete parent names the feed cut short (DOGWOOD STATE BANK A DIVISION OF TOWNEBAN → TOWNEBANK)
_known = [str(x).upper() for x in dim_names.fi.dropna()]
_clean = lambda x: re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", str(x).upper())).strip()
_div_tails = [fi_fix(x) for x in pd.concat([dim_names.fi, rt_raw.fi]).dropna().unique()
              if re.match(FI_DIVISION_RE, _clean(x))]
TRUNC_FIX = complete_truncated(_div_tails, _known)
rt_raw["fi_fixed"] = rt_raw.fi_fixed.replace(TRUNC_FIX)
dim_names["fi_fixed"] = dim_names.fi_fixed.replace(TRUNC_FIX)
rt_raw["key"] = rt_raw.fi_fixed.map(fi_key)
dim_names["key"] = dim_names.fi_fixed.map(fi_key)

show("§1 Inputs",
     ("Size", pd.DataFrame([
         {"item": "routing × month × FI rows", "value": f"{len(rt_raw):,}"},
         {"item": "routing numbers", "value": f"{rt_raw.routing.nunique():,}"},
         {"item": "months", "value": f"{rt_raw.month.min()} → {rt_raw.month.max()} ({rt_raw.month.nunique()})"},
         {"item": "distinct raw FI names under a routing", "value": f"{rt_raw.fi.nunique():,}"},
         {"item": f"distinct raw FI names in {T_CPTY_DIM}", "value": f"{len(dim_names):,}"},
         {"item": "spelling groups (normalised key)", "value": f"{dim_names.key.nunique():,}"}])))

## §2 Canonical name per routing number

- Canonical name of a routing number = its dominant name (by dollars) in the **latest** month it appears; when the
  routing carries several spellings of that name (same normalised key), the spelling with the most dollars on that
  routing. Spellings are never unified across routing numbers by the key alone.
- Fintech partner names are reduced to the bank first ("FIDELITY INVESTMENTS WITH UMB BANK" → "UMB BANK").
- Consistency: routing numbers that carry the **same set** of names (e.g. both Webster/Santander routings, all three
  Comerica/Fifth Third routings) get the same canonical name — the one with the most dollars in their latest month.
- Change type: `single name`; `spelling variant` (several raw names, one key); `multiple names` (merger or shared
  routing). `FI_KEEP_RAW_ROUTINGS` keeps listed routings un-unified (canonical left empty → name map / raw name).

In [ ]:
# §2  Build the routing table
# display spelling of each (routing, key): the spelling with the most dollars on that routing
rk = (rt_raw.dropna(subset=["key"]).groupby(["routing", "key", "fi_fixed"]).amt.sum().reset_index()
            .sort_values(["routing", "key", "amt"], ascending=[True, True, False]).drop_duplicates(["routing", "key"]))
RK_SPELL = {(r, k): sp for r, k, sp in zip(rk.routing, rk.key, rk.fi_fixed)}

last_m = rt_raw.groupby("routing").month.max().rename("last_month")
first_m = rt_raw.groupby("routing").month.min().rename("first_month")
latest = (rt_raw.merge(last_m, left_on=["routing", "month"], right_on=["routing", "last_month"])
                .sort_values(["routing", "amt"], ascending=[True, False]).drop_duplicates("routing"))
latest = latest.set_index("routing")

kspan = rt_raw.groupby(["routing", "key"]).agg(k_first=("month", "min"), k_last=("month", "max"), k_amt=("amt", "sum"))
rows = []
for rt, g in rt_raw.groupby("routing"):
    keys = g.key.dropna().unique()
    raw = sorted(g.fi.unique())
    lk = latest.loc[rt, "key"]
    canon = RK_SPELL.get((rt, lk), latest.loc[rt, "fi_fixed"])
    if len(raw) == 1:
        ctype, switch = "single name", None
    elif len(keys) == 1:
        ctype, switch = "spelling variant", None
    elif lk is None or lk not in kspan.loc[rt].index:
        ctype, switch = "concurrent names", None
    else:
        ks = kspan.loc[rt]
        new_first = ks.loc[lk, "k_first"]
        others_last = ks.drop(index=lk).k_last.max()
        ctype = "renamed / acquired" if others_last < new_first else "concurrent names"
        switch = new_first if ctype == "renamed / acquired" else None
    rows.append({"routing": rt, "fi_canonical": canon, "n_names": len(raw), "n_keys": len(keys),
                 "change_type": ctype, "switch_month": switch, "names_seen": " | ".join(raw[:6]),
                 "first_month": first_m[rt], "last_month": last_m[rt],
                 "ids": int(g.groupby("month").ids.max().max()), "amt": float(g.amt.sum())})
ROUT = pd.DataFrame(rows)
ROUT["change_type"] = ROUT.change_type.replace({"renamed / acquired": "multiple names", "concurrent names": "multiple names"})

# Consistency across routings with the same name set: one canonical per set (most $ in the set's latest month)
kset = rt_raw.dropna(subset=["key"]).groupby("routing").key.apply(lambda s: "|".join(sorted(set(s))))
ROUT["key_set"] = ROUT.routing.map(kset)
multi = ROUT[ROUT.change_type == "multiple names"]
for ks, grp in multi.groupby("key_set"):
    if len(grp) < 2:
        continue
    sub = rt_raw[rt_raw.routing.isin(grp.routing)]
    lm = sub.month.max()
    best = sub[sub.month == lm].groupby("key").amt.sum().idxmax()
    ROUT.loc[grp.index, "fi_canonical"] = sub[sub.key == best].groupby("fi_fixed").amt.sum().idxmax()

# Routings kept raw on request
ROUT["kept_raw"] = ROUT.routing.isin(FI_KEEP_RAW_ROUTINGS)
ROUT.loc[ROUT.kept_raw, "fi_canonical"] = None

ct = ROUT.groupby("change_type").agg(routing_numbers=("routing", "count"), amt=("amt", "sum")).reset_index()
ct["% routing numbers"] = 100 * ct.routing_numbers / ct.routing_numbers.sum()
ct["% $"] = 100 * ct.amt / ct.amt.sum()
top_change = (ROUT[ROUT.change_type != "single name"].sort_values("amt", ascending=False).head(25)
                  .assign(**{"$B": lambda d: d.amt / 1e9}))
show(f"§2 Routing numbers: {len(ROUT):,}",
     ("By change type", ct[["change_type", "routing_numbers", "% routing numbers", "% $"]]),
     ("Top 25 routing numbers with more than one name (by $)",
      top_change[["routing", "change_type", "fi_canonical", "names_seen", "$B"]]),
     note(f"Typo fixes applied: {FI_FIXES}. Add more to FI_FIXES and rerun from §1."))

In [ ]:
# §2b  Pairs to verify: multiple-name routings whose names share no distinctive word (not an obvious rename)
names_by_rt = rt_raw.groupby("routing").fi_fixed.apply(lambda s: sorted(set(s.dropna())))
def unrelated(names):
    toks = [fi_tokens(n) for n in names]
    return len(names) > 1 and all(not (a & b) for i, a in enumerate(toks) for b in toks[i + 1:])
ver = ROUT[(ROUT.change_type == "multiple names") & ROUT.routing.map(lambda r: unrelated(names_by_rt.get(r, [])))]
ver = ver.sort_values("amt", ascending=False).head(30).assign(**{"$B": lambda d: d.amt / 1e9})
show(f"§2b Routings whose names share no distinctive word: {len(ver):,} shown (verify; mergers are expected here too)",
     ("Check these; add unrelated ones to FI_KEEP_RAW_ROUTINGS and rerun from §2",
      ver[["routing", "fi_canonical", "names_seen", "ids", "$B", "kept_raw"]]),
     note("Most are real mergers with new brands (COMERICA → FIFTH THIRD). Unrelated pairs (e.g. two credit unions in "
          "different states) usually point to a reference-data error and should be kept raw."))

## §3 Raw name → canonical name

In [ ]:
# §3  Name map: each raw FI name → the canonical name of the routing numbers it appears under, when one canonical
#     name holds ≥ NAME_MIN_SHARE of its dollars there; otherwise the raw name (typo and partner fixes applied)
rt_c = rt_raw.merge(ROUT.loc[~ROUT.kept_raw, ["routing", "fi_canonical"]], on="routing")
by = rt_c.groupby(["fi", "fi_canonical"]).agg(amt=("amt", "sum"), routings=("routing", "nunique")).reset_index()
by["share"] = by.amt / by.groupby("fi").amt.transform("sum").clip(lower=1e-9)
by = by.sort_values(["fi", "amt"], ascending=[True, False])
alts = by.groupby("fi").apply(lambda g: " | ".join(f"{c} {100 * sh:.0f}%" for c, sh in zip(g.fi_canonical[:4], g.share[:4])))
top = by.drop_duplicates("fi")[["fi", "fi_canonical", "share"]].rename(columns={"fi_canonical": "top_canonical"})
top["n_canonical"] = top.fi.map(by.groupby("fi").fi_canonical.nunique())
top["canonicals_seen"] = top.fi.map(alts)
NAMEMAP = dim_names[["fi", "fi_fixed", "key", "ids"]].merge(top, on="fi", how="left")
clear = NAMEMAP.share >= NAME_MIN_SHARE
NAMEMAP["source"] = np.select([NAMEMAP.top_canonical.isna(), clear], ["no routing", "routing"], "ambiguous")
NAMEMAP["fi_canonical"] = np.where(clear, NAMEMAP.top_canonical, NAMEMAP.fi_fixed)
NAMEMAP["changed"] = NAMEMAP.fi != NAMEMAP.fi_canonical

summ = pd.DataFrame([
    {"item": "raw FI names", "value": f"{len(NAMEMAP):,}"},
    {"item": "canonical names", "value": f"{NAMEMAP.fi_canonical.nunique():,}"},
    {"item": "raw names mapped to a different name", "value": f"{NAMEMAP.changed.sum():,}"},
    {"item": "ids (dim) whose FI name changes", "value": f"{NAMEMAP.loc[NAMEMAP.changed, 'ids'].sum():,} of {NAMEMAP.ids.sum():,}"}])
groups = (NAMEMAP[NAMEMAP.changed].groupby("fi_canonical")
                 .agg(raw_names=("fi", lambda s: " | ".join(sorted(s)[:5])), n=("fi", "count"), ids=("ids", "sum"))
                 .reset_index().sort_values("ids", ascending=False).head(25))
src = NAMEMAP.groupby("source").agg(raw_names=("fi", "count"), ids=("ids", "sum"),
                                     changed=("changed", "sum")).reset_index()
amb = (NAMEMAP[NAMEMAP.source == "ambiguous"].sort_values("ids", ascending=False).head(20)
              [["fi", "ids", "n_canonical", "canonicals_seen"]])
show("§3 Name map", ("Summary", summ), ("By source", src),
     ("Largest renames (canonical ← raw names)", groups.rename(columns={"n": "raw names folded", "ids": "ids affected"})),
     (f"Ambiguous raw names (no canonical name with ≥ {100 * NAME_MIN_SHARE:.0f}% of their dollars): kept as is "
      "for ids without a routing number", amb),
     note("Ids with a routing number always take the canonical name of their routing (pkg_fi_routing); the name map is "
          "only the fallback for name-hash ids."))

## §4 Impact on bank metrics for one month

Per customer × direction in `IMPACT_MONTH`: distinct banks with raw names vs canonical names, all rails and ACH-only
outgoing (`fin_out_n`). Then month-over-month **lost banks** (bank in the previous month, not this month): with raw names
an acquisition or spelling flip shows as lost + new; with canonical names it does not. FI per counterparty is taken from
`pkg_cpty_dim`, as in the customer metrics build.

In [ ]:
# §4  Raw vs canonical bank counts and lost banks
M0, M1 = prev_month(IMPACT_MONTH), IMPACT_MONTH
rt_df = F.broadcast(spark.createDataFrame(ROUT.loc[~ROUT.kept_raw, ["routing", "fi_canonical"]])
                    .withColumnRenamed("fi_canonical", "fi_c_rt"))
nm_df = F.broadcast(spark.createDataFrame(NAMEMAP[["fi", "fi_canonical"]]).withColumnRenamed("fi_canonical", "fi_c_nm"))

e = (spark.table(T_EDGE_M).where(F.col("month").isin(M0, M1))
          .where((F.col("src_kind") == "CPTY") | (F.col("dst_kind") == "CPTY"))
          .select("month",
                  F.when(F.col("dst_kind") == "CPTY", F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                  F.when(F.col("dst_kind") == "CPTY", F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
                  F.when(F.col("dst_kind") == "CPTY", F.lit("out")).otherwise(F.lit("in")).alias("dir"),
                  F.substring("category", 1, 1).isin("1", "2", "3").alias("ach"))
          .distinct())
dim = spark.table(T_CPTY_DIM).select("cpty_id", F.col(DIM_FI).alias("fi")).where(F.col("fi").isNotNull())
cb = (e.join(dim, "cpty_id", "inner")
       .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
       .join(rt_df, "routing", "left").join(nm_df, "fi", "left")
       .withColumn("fi_c", F.coalesce("fi_c_rt", "fi_c_nm", "fi"))
       .select("month", "cust", "dir", "ach", "fi", "fi_c")
       .cache())

def counts(df, ach_only):
    d = df.where(F.col("ach")) if ach_only else df
    return d.groupBy("cust", "dir").agg(F.countDistinct("fi").alias("raw"), F.countDistinct("fi_c").alias("canon"))

cd = spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("cust"), F.col("is_tm").cast("boolean").alias("is_tm"))
rows = []
for label, ach_only, dirs in [("all rails, out", False, ["out"]), ("all rails, in", False, ["in"]),
                              ("ACH out (fin_out_n)", True, ["out"])]:
    c = counts(cb.where((F.col("month") == M1) & F.col("dir").isin(dirs)), ach_only).join(cd, "cust", "left")
    r = (c.groupBy(F.coalesce("is_tm", F.lit(False)).alias("tm"))
          .agg(F.count("*").alias("customers"),
               F.avg((F.col("raw") != F.col("canon")).cast("double")).alias("chg"),
               F.avg(F.col("raw") - F.col("canon")).alias("mean_diff"),
               F.max(F.col("raw") - F.col("canon")).alias("max_diff"),
               F.avg("raw").alias("mean_raw")).toPandas())
    for _, x in r.iterrows():
        rows.append({"metric": label, "customers": "TM" if x.tm else "non-TM", "n": x.customers,
                     "% customers whose count changes": 100 * x.chg, "mean banks (raw)": x.mean_raw,
                     "mean overcount": x.mean_diff, "max overcount": x.max_diff})
cnt_tab = pd.DataFrame(rows)

# lost banks M0 → M1, per customer × direction
def lost(col):
    p0 = cb.where(F.col("month") == M0).select("cust", "dir", F.col(col).alias("b")).distinct()
    p1 = cb.where(F.col("month") == M1).select("cust", "dir", F.col(col).alias("b")).distinct()
    active = p1.select("cust", "dir").distinct()
    return (p0.join(p1, ["cust", "dir", "b"], "left_anti").join(active, ["cust", "dir"], "inner")
              .groupBy("cust", "dir").agg(F.count("*").alias(f"lost_{col}")))
L = (lost("fi").join(lost("fi_c"), ["cust", "dir"], "full").fillna(0, ["lost_fi", "lost_fi_c"]))
lt = (L.groupBy("dir").agg(F.sum((F.col("lost_fi") > 0).cast("int")).alias("with lost bank (raw)"),
                           F.sum((F.col("lost_fi_c") > 0).cast("int")).alias("with lost bank (canonical)"),
                           F.sum("lost_fi").alias("lost banks (raw)"), F.sum("lost_fi_c").alias("lost banks (canonical)"),
                           F.sum(((F.col("lost_fi") > 0) & (F.col("lost_fi_c") == 0)).cast("int")).alias("false lost only"))
        .toPandas())
lt["% lost events that are false"] = 100 * (1 - lt["lost banks (canonical)"] / lt["lost banks (raw)"].clip(lower=1))

# which raw names produce the false lost events
p0 = cb.where(F.col("month") == M0).select("cust", "dir", "fi", "fi_c").distinct()
p1 = cb.where(F.col("month") == M1).select("cust", "dir", "fi", "fi_c").distinct()
false_lost = (p0.join(p1.select("cust", "dir", "fi"), ["cust", "dir", "fi"], "left_anti")
                .join(p1.select("cust", "dir", "fi_c").distinct(), ["cust", "dir", "fi_c"], "inner")
                .groupBy("fi", "fi_c").agg(F.count("*").alias("false lost events"))
                .orderBy(F.desc("false lost events")).limit(20).toPandas())
show(f"§4 Impact, {M1} (lost banks vs {M0})",
     ("Distinct banks per customer: raw vs canonical", cnt_tab),
     ("Lost banks month over month (customers active in both months)", lt),
     ("Raw names behind the false lost events (raw name lost, canonical bank still there)",
      false_lost.rename(columns={"fi": "raw name lost", "fi_c": "canonical bank (still present)"})))

## §4b All months: false bank churn removed, and bank swaps still left

§4 looks at one month; renames happen in different months. Here every month from the first to the last is checked,
for customer × direction links (TM customers by default) with the **monthly** FI of each counterparty
(`pkg_cpty_month`, as the counterparty metrics use).

1. **By month:** lost banks (a bank the customer link had last month and not this month) with raw names and with
   canonical names. A month where the canonical count stays high is a month with something left to fix.
2. **Largest false-lost sources:** raw names whose loss disappears with canonical names (what the mapping fixes).
3. **Swaps still left:** customer links that lose bank X and gain bank Y in the same month, counted per (month, X, Y).
   When X's customer links collapse that month and most of them move to Y, it is a rename the routing numbers do not
   explain (accounts moved to new routing numbers, or ids without a routing number). Check the list; for real renames
   add `"X": "Y"` to `FI_FIXES` (a snippet is printed) and rerun from §1. Pairs that are real customers changing banks
   stay as they are.

The customer × bank history is cached once per month under the database's scratch folder (raw names only, so changing
`FI_FIXES` or `FI_KEEP_RAW_ROUTINGS` does not rescan the edges). An interrupted run continues with the missing months.

In [ ]:
# §4b-1  Customer link × month × (routing, raw FI) history, cached per month (resumable)
def _fs(p):
    path = spark._jvm.org.apache.hadoop.fs.Path(p)
    return path.getFileSystem(spark._jsc.hadoopConfiguration()), path
def exists(p):
    fs, path = _fs(p)
    return fs.exists(path)
def delete(p):
    fs, path = _fs(p)
    return bool(fs.exists(path) and fs.delete(path, True))
LOC = next(str(r[1]).rstrip("/") for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {DB}").collect()
           if str(r[0]).strip().lower() == "location")
P_HIST = f"{LOC}/_tmp_pkg_fi_canonical/hist_{'tm' if HIST_TM_ONLY else 'all'}"
HMONTHS = sorted(rt_raw.month.unique())
if HIST_REBUILD:
    delete(P_HIST)
cust_keep = (spark.table(T_CUST_DIM).where(F.col("is_tm").cast("boolean")) if HIST_TM_ONLY
             else spark.table(T_CUST_DIM)).select(F.col("mdm_id").alias("cust"))
t0, built = time.time(), []
for m in HMONTHS:
    out = f"{P_HIST}/month={m}"
    if exists(out + "/_SUCCESS"):
        continue
    to_c = F.col("dst_kind") == "CPTY"
    e = (spark.table(T_EDGE_M).where(F.col("month") == m).where((F.col("src_kind") == "CPTY") | to_c)
              .select(F.when(to_c, F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                      F.when(to_c, F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
                      F.when(to_c, F.lit("out")).otherwise(F.lit("in")).alias("dir"))      # customer's view
              .distinct()
              .join(cust_keep, "cust", "left_semi"))
    fim = (spark.table(T_CPTY_MONTH).where((F.col("month") == m) & F.col("fi").isNotNull())
                .groupBy("cpty_id").agg(F.expr("max_by(fi, amount)").alias("fi")))
    (e.join(fim, "cpty_id", "inner")
      .select("cust", "dir", F.regexp_extract("cpty_id", r"^(\d{9})-", 1).alias("routing"), "fi")
      .distinct()
      .write.mode("overwrite").parquet(out))
    built.append(m)
show(f"§4b-1 History cache: {len(HMONTHS)} months ({HMONTHS[0]} → {HMONTHS[-1]}), "
     f"{len(built)} built this run in {(time.time() - t0) / 60:.1f} min",
     note(f"{P_HIST} — customers: {'TM only' if HIST_TM_ONLY else 'all'}. Delete with HIST_REBUILD = True "
          "after the edge tables or pkg_cpty_month are rebuilt."))

In [ ]:
# §4b-2  By month, false-lost sources, and swaps still left after canonical names
rt_b = F.broadcast(spark.createDataFrame(ROUT.loc[~ROUT.kept_raw, ["routing", "fi_canonical"]])
                   .withColumnRenamed("fi_canonical", "fi_c_rt"))
nm_b = F.broadcast(spark.createDataFrame(NAMEMAP[["fi", "fi_canonical"]]).withColumnRenamed("fi_canonical", "fi_c_nm"))
H = (spark.read.option("basePath", P_HIST).parquet(*[f"{P_HIST}/month={m}" for m in HMONTHS])
          .join(rt_b, "routing", "left").join(nm_b, "fi", "left")
          .withColumn("fi_c", F.coalesce("fi_c_rt", "fi_c_nm", "fi"))
          .withColumn("mi", F.substring("month", 1, 4).cast("int") * 12 + F.substring("month", 6, 2).cast("int") - 1)
          .select("mi", "cust", "dir", "fi", "fi_c").distinct())
P_PR = f"{LOC}/_tmp_pkg_fi_canonical/presence"
H.write.mode("overwrite").parquet(P_PR)
pr = spark.read.parquet(P_PR)
K = ["mi", "cust", "dir"]
active = pr.select(*K).distinct()
mstr = lambda c: F.concat(F.floor(F.col(c) / 12).cast("string"), F.lit("-"), F.lpad(((F.col(c) % 12) + 1).cast("string"), 2, "0"))

def lost_by_month(col):
    p = pr.select(*K, F.col(col).alias("b")).distinct()
    prev = p.withColumn("mi", F.col("mi") + 1)
    return (prev.join(p, K + ["b"], "left_anti").join(active, K, "inner")
                .groupBy("mi").agg(F.count("*").alias(col)))
bym = (active.groupBy("mi").agg(F.count("*").alias("customer links"))
             .join(lost_by_month("fi"), "mi", "left").join(lost_by_month("fi_c"), "mi", "left")
             .withColumn("month", mstr("mi")).orderBy("mi").toPandas().fillna(0))
bym = bym[bym.mi > bym.mi.min()].rename(columns={"fi": "lost (raw)", "fi_c": "lost (canonical)"})
bym["% of lost that is false"] = 100 * (1 - bym["lost (canonical)"] / bym["lost (raw)"].clip(lower=1))
bym["lost (canonical) per 1k links"] = 1000 * bym["lost (canonical)"] / bym["customer links"].clip(lower=1)

# raw names whose loss is false (the canonical bank is still there), all months
pfc = pr.select(*K, "fi", "fi_c").distinct()
lost_raw = (pfc.withColumn("mi", F.col("mi") + 1).join(pfc.select(*K, "fi"), K + ["fi"], "left_anti")
               .join(active, K, "inner"))
false_src = (lost_raw.join(pfc.select(*K, "fi_c").distinct(), K + ["fi_c"], "inner")
                     .groupBy("fi", "fi_c", "mi").agg(F.count("*").alias("n"))
                     .groupBy("fi", "fi_c").agg(F.sum("n").alias("false lost events"),
                                                F.count("*").alias("months"),
                                                F.expr("max_by(mi, n)").alias("pk"))
                     .withColumn("peak month", mstr("pk")).drop("pk")
                     .orderBy(F.desc("false lost events")).limit(25).toPandas())

# swaps X → Y still left with canonical names
pc = pr.select(*K, F.col("fi_c").alias("b")).distinct()
prevc = pc.withColumn("mi", F.col("mi") + 1)
cur_links = active
prev_links = active.withColumn("mi", F.col("mi") + 1)
lost_c = prevc.join(pc, K + ["b"], "left_anti").join(cur_links, K, "inner")
gain_c = pc.join(prevc, K + ["b"], "left_anti").join(prev_links, K, "inner")
small = lambda d: d.groupBy(*K).agg(F.count("*").alias("n")).where(F.col("n") <= SWAP_MAX_CHANGES).select(*K)
pairs = (lost_c.join(small(lost_c), K, "left_semi").withColumnRenamed("b", "X")
               .join(gain_c.join(small(gain_c), K, "left_semi").withColumnRenamed("b", "Y"), K, "inner")
               .groupBy("mi", "X", "Y").agg(F.count("*").alias("links swapping")))
base = pc.groupBy("mi", "b").agg(F.count("*").alias("links"))
sw = (pairs.where(F.col("links swapping") >= SWAP_MIN_CUST)
           .join(base.withColumn("mi", F.col("mi") + 1).withColumnRenamed("b", "X")
                     .withColumnRenamed("links", "X links last month"), ["mi", "X"], "left")
           .join(base.withColumnRenamed("b", "X").withColumnRenamed("links", "X links this month"), ["mi", "X"], "left")
           .fillna(0, ["X links this month"])
           .withColumn("share", F.col("links swapping") / F.col("X links last month"))
           .where(F.col("share") >= SWAP_MIN_SHARE)
           .withColumn("month", mstr("mi"))
           .orderBy(F.desc("links swapping")).limit(40).toPandas())
if len(sw):
    sw["% of X's links swapping"] = 100 * sw.share
    sw["X kept % of links"] = 100 * sw["X links this month"] / sw["X links last month"].clip(lower=1)
    sw["shared word"] = [bool(fi_tokens(x) & fi_tokens(y)) for x, y in zip(sw.X, sw.Y)]
    top_y = ~sw.duplicated(["month", "X"])                 # sw is sorted by links swapping: first Y = X's main destination
    sw["rename-like"] = (sw["X kept % of links"] <= 20) & top_y
    snippet = {}
    for x, y, r in zip(sw.X, sw.Y, sw["rename-like"]):
        if r and x not in snippet:
            snippet[x] = y
else:
    snippet = {}
show(f"§4b-2 All months ({'TM' if HIST_TM_ONLY else 'all'} customer links)",
     ("By month: lost banks, raw vs canonical",
      bym[["month", "customer links", "lost (raw)", "lost (canonical)", "% of lost that is false",
           "lost (canonical) per 1k links"]]),
     ("Largest false-lost sources over all months (raw name lost, canonical bank still there)",
      false_src.rename(columns={"fi": "raw name lost", "fi_c": "canonical bank (still present)"})),
     (f"Swaps still left with canonical names (≥ {SWAP_MIN_CUST} links and ≥ {100 * SWAP_MIN_SHARE:.0f}% of X's links "
      "in one month)", sw[["month", "X", "Y", "links swapping", "X links last month", "X links this month",
                           "% of X's links swapping", "X kept % of links", "shared word", "rename-like"]]
      if len(sw) else pd.DataFrame([{"result": "no swaps above the thresholds"}])),
     note("rename-like = X kept ≤ 20% of its links that month (the old name disappears) and Y is where most of them went. Check each one; real renames "
          "go into FI_FIXES, then rerun from §1. §4b-3 shows which counterparties are behind each swap."))

In [ ]:
# §4b-3  Who is behind each swap: one counterparty moving its account (real, leave it) or a bank renamed (fix it)
#        For each listed swap, the counterparty names on the customer links that swapped, on both sides.
def month_links(m, banks):
    # customer links × counterparty at the given banks in month m (Spark DataFrame, canonical bank names)
    to_c = F.col("dst_kind") == "CPTY"
    e = (spark.table(T_EDGE_M).where(F.col("month") == m).where((F.col("src_kind") == "CPTY") | to_c)
              .select(F.when(to_c, F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                      F.when(to_c, F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
                      F.when(to_c, F.lit("out")).otherwise(F.lit("in")).alias("dir")).distinct()
              .join(cust_keep, "cust", "left_semi"))
    fim = (spark.table(T_CPTY_MONTH).where((F.col("month") == m) & F.col("fi").isNotNull())
                .groupBy("cpty_id").agg(F.expr("max_by(fi, amount)").alias("fi"),
                                        F.expr("max_by(cpty_name, amount)").alias("cpty_name")))
    return (e.join(fim, "cpty_id", "inner")
             .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
             .join(rt_b, "routing", "left").join(nm_b, "fi", "left")
             .withColumn("fi_c", F.coalesce("fi_c_rt", "fi_c_nm", "fi"))
             .where(F.col("fi_c").isin(banks))
             .select("cust", "dir", "fi_c", "cpty_name"))

def top_name(df):
    if not len(df):
        return None, 0.0
    vc = df.drop_duplicates(["cust", "dir", "cpty_name"]).cpty_name.value_counts()
    return vc.index[0], vc.iloc[0] / df[["cust", "dir"]].drop_duplicates().shape[0]

rows, t0 = [], time.time()
for (mon, mi), grp in (sw.groupby(["month", "mi"]) if len(sw) else []):
    prev_m = f"{(mi - 1) // 12}-{(mi - 1) % 12 + 1:02d}"
    banks = sorted(set(grp.X) | set(grp.Y))
    a_s = month_links(prev_m, banks)
    xcust = a_s.where(F.col("fi_c").isin(sorted(set(grp.X)))).select("cust", "dir").distinct()
    a = a_s.join(xcust, ["cust", "dir"], "left_semi").toPandas()          # only links that had an X bank
    b = month_links(mon, banks).join(xcust, ["cust", "dir"], "left_semi").toPandas()
    for _, r in grp.iterrows():
        xa = a[a.fi_c == r.X][["cust", "dir"]].drop_duplicates()
        xb = b[b.fi_c == r.X][["cust", "dir"]].drop_duplicates()
        ya = a[a.fi_c == r.Y][["cust", "dir"]].drop_duplicates()
        sw_links = (b[b.fi_c == r.Y][["cust", "dir"]].drop_duplicates()
                    .merge(xa, on=["cust", "dir"]).merge(xb, on=["cust", "dir"], how="left", indicator=True)
                    .query("_merge == 'left_only'").drop(columns="_merge")
                    .merge(ya, on=["cust", "dir"], how="left", indicator=True)
                    .query("_merge == 'left_only'").drop(columns="_merge"))
        before = a[a.fi_c == r.X].merge(sw_links, on=["cust", "dir"])
        after = b[b.fi_c == r.Y].merge(sw_links, on=["cust", "dir"])
        nb, sb = top_name(before)
        na, sa = top_name(after)
        same = nb is not None and na is not None and fi_tokens(nb) & fi_tokens(na) or nb == na
        if (sa >= 0.5 or sb >= 0.5) and same:
            verdict = "one counterparty moved its account (real; leave)"
        elif r["X kept % of links"] <= 20 and sa < 0.5:
            verdict = "bank renamed (add to FI_FIXES)"
        else:
            verdict = "mixed (check)"
        rows.append({"month": mon, "X": r.X, "Y": r.Y, "links swapping": r["links swapping"],
                     "top counterparty before (at X)": nb, "% links before": 100 * sb,
                     "top counterparty after (at Y)": na, "% links after": 100 * sa,
                     "counterparties after": after.cpty_name.nunique(), "verdict": verdict})
who = pd.DataFrame(rows)
fix = {r.X: r.Y for r in who.itertuples() if r.verdict.startswith("bank renamed")} if len(who) else {}
show(f"§4b-3 Who is behind each swap ({(time.time() - t0) / 60:.1f} min)",
     ("Swaps with the counterparties behind them", who) if len(who) else None,
     note(f"Division names completed from truncation: {TRUNC_FIX or 'none'}."),
     note("Lines for FI_FIXES (verdict = bank renamed): "
          + (", ".join(f'"{x}": "{y}"' for x, y in fix.items()) if fix else "none")))

## §5 Write the reference tables

In [ ]:
# §5  Write pkg_fi_routing and pkg_fi_name_map (small, non-partitioned) and print the downstream join
if WRITE_TABLES:
    rt_out = ROUT.drop(columns=["amt", "switch_month"]).assign(amount=ROUT.amt)
    nm_out = NAMEMAP[["fi", "fi_canonical", "source", "ids"]].rename(columns={"fi": "fi_raw"})
    def to_sdf(pdf):
        # explicit schema; NaN → NULL in text columns (avoids type-inference failures on sparse columns)
        typ = {"object": "string", "bool": "boolean", "int64": "bigint", "int32": "int", "float64": "double"}
        pdf = pdf.copy()
        for c in pdf.columns:
            if pdf[c].dtype == object:
                pdf[c] = pdf[c].astype(object).where(pdf[c].notna(), None)
        schema = ", ".join(f"`{c}` {typ.get(str(pdf[c].dtype), 'string')}" for c in pdf.columns)
        return spark.createDataFrame(pdf, schema)
    for t, pdf in [(T_FI_ROUTING, rt_out), (T_FI_NAMEMAP, nm_out)]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        to_sdf(pdf).coalesce(1).write.mode("overwrite").format("parquet").saveAsTable(t)
    written = pd.DataFrame([{"table": T_FI_ROUTING, "rows": len(rt_out)}, {"table": T_FI_NAMEMAP, "rows": len(nm_out)}])
else:
    written = pd.DataFrame([{"table": "(not written: WRITE_TABLES = False)", "rows": 0}])
show("§5 Reference tables", ("Written", written),
     note("Canonical FI per counterparty: "
          "coalesce(pkg_fi_routing.fi_canonical via regexp_extract(cpty_id, '^(\\\\d{9})-', 1), "
          "pkg_fi_name_map.fi_canonical via the raw FI name, raw FI name)."),
     note(f"Impala: INVALIDATE METADATA {T_FI_ROUTING}; INVALIDATE METADATA {T_FI_NAMEMAP};"))